In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor

# 학습 모델 저장을 위한 라이브러리
import pickle

# 데이터 준비

In [3]:
# parquet 파일 읽기
train_df = pd.read_parquet('/content/drive/MyDrive/파이널 프로젝트/1.회원정보_train_AB_CDE_final_dataset.parquet')
test_df = pd.read_parquet('/content/drive/MyDrive/파이널 프로젝트/1_회원정보_test_AB_CDE_final_dataset.parquet')


# 데이터 확인
display(train_df)
display(test_df)

,입회경과개월수_신용,이용금액_R3M_신용,_1순위카드이용건수,이용금액_R3M_신용_가족,최종탈회후경과월,이용카드수_신용,Group
0,67,196,26,0,61,1,CDE
1,12,13475,46,0,98,1,CDE
2,124,23988,28,0,60,1,CDE
3,27,3904,1,0,100,1,CDE
4,2,0,-2,0,101,0,CDE
...,...,...,...,...,...,...,...
2399995,209,7267,3,0,0,0,CDE
2399996,17,27636,38,0,50,1,CDE
2399997,115,23187,33,0,0,1,CDE
2399998,71,0,-2,0,0,0,CDE


,입회경과개월수_신용,이용금액_R3M_신용,_1순위카드이용건수,이용금액_R3M_신용_가족,최종탈회후경과월,이용카드수_신용
0,51,21458,51,7879,100,2
1,30,10759,40,0,20,1
2,5,40758,154,0,101,2
3,73,5255,105,0,0,1
4,176,14290,52,0,0,2
...,...,...,...,...,...,...
599995,69,0,-2,0,1,0
599996,14,3110,4,0,0,1
599997,6,0,6,0,23,0
599998,82,113786,185,0,65,4


In [5]:
# 데이터 프레임을 합친다.
all_df = pd.concat([train_df, test_df])
all_df.reset_index(inplace=True, drop=True)
all_df

,입회경과개월수_신용,이용금액_R3M_신용,_1순위카드이용건수,이용금액_R3M_신용_가족,최종탈회후경과월,이용카드수_신용,Group
0,67,196,26,0,61,1,CDE
1,12,13475,46,0,98,1,CDE
2,124,23988,28,0,60,1,CDE
3,27,3904,1,0,100,1,CDE
4,2,0,-2,0,101,0,CDE
...,...,...,...,...,...,...,...
2999995,69,0,-2,0,1,0,NaN
2999996,14,3110,4,0,0,1,NaN
2999997,6,0,6,0,23,0,NaN
2999998,82,113786,185,0,65,4,NaN


In [6]:
# 결과 데이터는 제거한다.
all_df.drop('Group', axis=1, inplace=True)
all_df

,입회경과개월수_신용,이용금액_R3M_신용,_1순위카드이용건수,이용금액_R3M_신용_가족,최종탈회후경과월,이용카드수_신용
0,67,196,26,0,61,1
1,12,13475,46,0,98,1
2,124,23988,28,0,60,1
3,27,3904,1,0,100,1
4,2,0,-2,0,101,0
...,...,...,...,...,...,...
2999995,69,0,-2,0,1,0
2999996,14,3110,4,0,0,1
2999997,6,0,6,0,23,0
2999998,82,113786,185,0,65,4


In [7]:
# Scaler 학습
scalerX = StandardScaler()
scalerX.fit(all_df)

StandardScaler()

In [8]:
# 라벨 인코더 생성
le = LabelEncoder()

# 문자열 y를 숫자로 변환
train_df['Group_encoded'] = le.fit_transform(train_df['Group'])

In [9]:
# 입력과 결과로 나눈다.
X = train_df.drop(['Group', 'Group_encoded'], axis=1)  # 또는 원하는 feature만 사용
y = train_df['Group_encoded']

In [10]:
# 표준화
X2 = scalerX.transform(X)
X2

array([[-0.09076952, -0.68813199, -0.26154312, -0.15850732,  0.87978951,
        -0.19290226],
       [-0.85059458, -0.08230067,  0.13034239, -0.15850732,  1.84535183,
        -0.19290226],
       [ 0.69668554,  0.3973367 , -0.22235457, -0.15850732,  0.85369324,
        -0.19290226],
       ...,
       [ 0.57235053,  0.36079247, -0.12438319, -0.15850732, -0.71208348,
        -0.19290226],
       [-0.03550952, -0.69707415, -0.81018283, -0.15850732, -0.71208348,
        -1.30411597],
       [-0.91966959,  0.28213796, -0.81018283, -0.15850732, -0.71208348,
         0.91831146]])

In [11]:
train_X = X2
train_y = y

In [12]:
# 학습이 완료된 모델을 저장할 파일 이름
best_model_path = '/content/drive/MyDrive/파이널 프로젝트/model/AB_CDE_신용카드데이터.dat'
# 교차검증 횟수
cv_count = 5
# 교차 검증
kfold = KFold(n_splits=cv_count, shuffle=True, random_state=1)
# 평가 결과를 담을 리스트
# 필요하다면 다른 것도 만들어주세요
f1_score_list = []
# 학습 모델 이름
model_name_list = []

# 학습

In [13]:
from sklearn.metrics import f1_score, make_scorer\

# XGBoost
xgb_basic_model = XGBClassifier(
    tree_method='gpu_hist',
    predictor='gpu_predictor',
    n_jobs=-1,
    use_label_encoder=False,
    eval_metric='logloss',
    verbosity=0
)

# F1 micro scorer 생성
f1_micro = make_scorer(f1_score, average='micro')

# 교차 검증
r1 = cross_val_score(xgb_basic_model, train_X, train_y, scoring=f1_micro, cv=kfold)

# 평가 결과
f1_score_list.append(r1.mean())
model_name_list.append("XGBoost Basic_f1_micro")

print("평균 F1 score:", r1.mean())

평균 F1 score: 0.9995508333333334


In [15]:
lgbm_basic_model = LGBMClassifier(device='gpu', verbose=-1, random_state=42)
# 교차 검증을 수행한다
r1_lgbm = cross_val_score(lgbm_basic_model, train_X, train_y, scoring='f1_micro', cv=kfold)
# 평가 결과를 담아준다
f1_score_list.append(r1_lgbm.mean())
# 학습 모델 이름을 담아준다
model_name_list.append("LGBM GPU_Basic_f1_micro")

In [17]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 22.8 MB/s eta 0:00:00


In [18]:
from catboost import CatBoostClassifier

# CatBoost 기본 모델 정의 (GPU 사용하려면 task_type='GPU' 옵션 추가)
cat_basic_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=7,
    task_type='GPU',     # GPU 사용 (CPU만 쓰려면 이 줄 빼면 됨)
    verbose=0,
    random_state=42
)

# 교차 검증
r1 = cross_val_score(cat_basic_model, train_X, train_y, scoring='f1_micro', cv=kfold)

# 평가 결과를 담는다
f1_score_list.append(r1.mean())

# 학습 모델 이름을 담는다
model_name_list.append("CatBoost Basic_f1_micro")

In [19]:
d1 = {
    'f1 score' : f1_score_list
}
result_df = pd.DataFrame(d1, index=model_name_list)
result_df.sort_values(by='f1 score', ascending=False, inplace=True)
result_df

,f1 score
CatBoost Basic_f1_micro,0.999560
XGBoost Basic_f1_micro,0.999551
LGBM GPU_Basic_f1_micro,0.998188


In [23]:
# 예측용 데이터 준비
X_all_scaled = scalerX.transform(all_df)

# 테스트 데이터 인덱스 지정 (ex: 마지막 N개)
test_index = test_df.index + len(train_df)  # test_df의 기존 인덱스를 train_df 뒤로 맞춘 것

# 테스트 데이터만 선택
X_test_final = X_all_scaled[-len(test_df):]

# 예측
preds = final_model.predict(X_test_final)

print(preds[:10])  # 예측 결과 일부 확인



[1 1 1 1 1 1 1 1 1 1]


In [25]:
test_df_copy = test_df.copy()
test_df_copy['predicted_Group'] = preds

display(test_df_copy.head(50))

,입회경과개월수_신용,이용금액_R3M_신용,_1순위카드이용건수,이용금액_R3M_신용_가족,최종탈회후경과월,이용카드수_신용,predicted_Group
0,51,21458,51,7879,100,2,1
1,30,10759,40,0,20,1,1
2,5,40758,154,0,101,2,1
3,73,5255,105,0,0,1,1
4,176,14290,52,0,0,2,1
5,58,0,-2,0,98,0,1
6,2,40632,138,0,0,1,1
7,2,0,-2,0,20,0,1
8,151,9130,-1,0,0,2,1
9,107,13445,27,0,0,1,1


In [26]:
print(le.classes_)

['AB' 'CDE']


In [27]:
# predicted_Group 컬럼이 0인 행만 추출
zero_group_df = test_df_copy[test_df_copy['predicted_Group'] == 0]

# 확인
display(zero_group_df.head())


,입회경과개월수_신용,이용금액_R3M_신용,_1순위카드이용건수,이용금액_R3M_신용_가족,최종탈회후경과월,이용카드수_신용,predicted_Group
22181,249,110878,178,0,0,1,0
38321,298,108918,197,11039,0,3,0
63557,282,98694,99,0,0,1,0
67564,271,87111,147,0,0,1,0
92601,226,120440,179,10969,0,2,0
